# GameTheory 2d — IESDS et rationalisabilité : élaguer un jeu, puis mesurer ce qui reste

**Navigation** : [<< 2-NormalForm (track principal)](GameTheory-02-NormalForm-Python.ipynb) · [2b-Lean-Definitions](GameTheory-02b-Lean-Definitions-Lean.ipynb) | [Index](README.md) | [4-NashEquilibrium >>](GameTheory-04-NashEquilibrium-Python.ipynb)

**Kernel** : `python3` — aucun appel réseau.

## Pourquoi ce carnet existe

La série pose le jeu en forme normale (`GameTheory-02-NormalForm-Python`), enchaîne sur la topologie des jeux 2×2 (`GameTheory-03-Topology2x2-Python`) puis arrive directement à l'équilibre de Nash (`GameTheory-04-NashEquilibrium-Python`). Entre les deux, il manque **le concept de solution qui les relie** : la **rationalisabilité** — et, en pratique, l'**élimination itérée des stratégies strictement dominées** (IESDS).

**Ce que le carnet 02 fait déjà** — on ne le refait pas : il définit la dominance stricte et implémente une IESDS sur un jeu 3×3 (section 6 de ce carnet-là).

**Ce qu'il ne fait pas** — et que ce carnet fait :

1. justifier que le résultat de l'IESDS **ne dépend pas de l'ordre** d'élimination (théorème pour la dominance stricte, **contre-exemple mesuré** pour la dominance faible) ;
2. distinguer l'IESDS (éliminer ce qui est *dominé*) de l'**admissibilité** (éliminer ce qui n'est *jamais meilleure réponse*) ;
3. relier les deux notions de solution par l'inclusion `rationalisable ⊇ Nash`, et exhiber un profil rationalisable qui **n'est pas** un équilibre de Nash.

## Plan

1. Briques : le module partagé `game_theory_utils.py`, et les fonctions de dominance
2. L'IESDS pas-à-pas, avec son journal d'éliminations
3. L'indépendance à l'ordre : mesure exhaustive sur 4096 jeux 2×3, et contre-exemple en dominance faible
4. L'admissibilité et les stratégies rationalisables
5. `Nash ⊆ rationalisable` : la mesure exhaustive, et un exemple de divergence
6. Exercices

## Prérequis

`GameTheory-02-NormalForm-Python` (forme normale, dominance, meilleure réponse) et, pour la version formelle des mêmes notions, `GameTheory-02b-Lean-Definitions-Lean`. L'équilibre de Nash lui-même est repris de `GameTheory-04-NashEquilibrium-Python` : ici on ne fait que le comparer aux ensembles rationalisables.

In [1]:
# Imports et localisation du module partage de la serie
import itertools
import pathlib
import sys

import numpy as np


def _dossier_serie():
    '''Dossier de la serie GameTheory (celui qui contient game_theory_utils.py).

    On cherche depuis le repertoire courant et ses parents : le carnet
    fonctionne ainsi qu'on l'execute depuis la racine du depot ou depuis
    le dossier de la serie.
    '''
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (base / "game_theory_utils.py").is_file():
            return base
    return None


DOSSIER_SERIE = _dossier_serie()
if DOSSIER_SERIE is None:
    raise FileNotFoundError(
        "game_theory_utils.py introuvable : executer ce carnet depuis "
        "MyIA.AI.Notebooks/GameTheory ou depuis la racine du depot"
    )
if str(DOSSIER_SERIE) not in sys.path:
    sys.path.insert(0, str(DOSSIER_SERIE))

# Briques deja ecrites par la serie : on les utilise, on ne les reecrit pas.
from game_theory_utils import NormalFormGame, find_pure_nash

print("Dossier de la serie :", DOSSIER_SERIE.name)
print("Classe importee    :", NormalFormGame.__module__ + ".NormalFormGame")

Dossier de la serie : GameTheory
Classe importee    : game_theory_utils.NormalFormGame


## 1. La dominance stricte, et ce qu'elle ne dit pas

Une stratégie $s_i$ **domine strictement** une stratégie $s_i'$ si le joueur $i$ fait **strictement mieux** avec $s_i$ qu'avec $s_i'$, **quelle que soit** la stratégie des autres :

$$u_i(s_i, s_{-i}) > u_i(s_i', s_{-i}) \quad \forall s_{-i} \in S_{-i}$$

Deux points qu'il faut garder en tête avant d'en déduire :

- la dominance est **stricte** : si l'inégalité est partout large et égale quelque part, on parle de dominance *faible* — et la section 3 montrera que les deux ne se comportent pas du tout pareil ;
- la dominance se lit **joueur par joueur** : une stratégie de la Ligne ne peut être éliminée que si *la Ligne* est mieux servie par une autre de ses stratégies.

Le module `game_theory_utils.py` fournit la représentation des jeux (`NormalFormGame`) et la recherche des équilibres de Nash purs (`find_pure_nash`). Les fonctions de dominance ci-dessous sont **nouvelles** : le module partagé ne les contient pas.

In [2]:
# Fonctions de dominance stricte, orientees par joueur
def dominance_stricte_lignes(A):
    '''A[m, n] : gains du joueur Ligne (les lignes sont SES strategies).

    Renvoie un dict {ligne -> index de la strategie qui la domine, ou None}.
    Une ligne est strictement dominee si une AUTRE ligne la bat sur
    TOUTES les colonnes.
    '''
    dominateurs = {}
    for i in range(A.shape[0]):
        dominateur = None
        for k in range(A.shape[0]):
            if k != i and np.all(A[k, :] > A[i, :]):
                dominateur = k
                break
        dominateurs[i] = dominateur
    return dominateurs


def dominance_stricte_colonnes(B):
    '''B[m, n] : gains du joueur Colonne (les colonnes sont SES strategies).

    Meme logique que pour les lignes, mais en transposant : pour le joueur
    Colonne, les strategies de l'adversaire sont les lignes.
    '''
    Bt = B.T
    dominateurs = {}
    for j in range(B.shape[1]):
        dominateur = None
        for l in range(B.shape[1]):
            if l != j and np.all(Bt[l, :] > Bt[j, :]):
                dominateur = l
                break
        dominateurs[j] = dominateur
    return dominateurs


# Verification sur un jeu canonique : le Dilemme du Prisonnier
IPD_A = np.array([[3, 0], [5, 1]], dtype=float)
IPD_B = np.array([[3, 5], [0, 1]], dtype=float)

dom_lignes = dominance_stricte_lignes(IPD_A)
dom_colonnes = dominance_stricte_colonnes(IPD_B)
print("Ligne 0 (Cooperer) dominee par :", dom_lignes[0])
print("Ligne 1 (Defaire) dominee par :", dom_lignes[1])
print("Colonne 0 (Cooperer) dominee par :", dom_colonnes[0])
print("Colonne 1 (Defaire) dominee par :", dom_colonnes[1])

Ligne 0 (Cooperer) dominee par : 1
Ligne 1 (Defaire) dominee par : None
Colonne 0 (Cooperer) dominee par : 1
Colonne 1 (Defaire) dominee par : None


### Lecture du résultat

Les quatre lignes affichent `1` ou `None` : `Cooperer` est strictement dominée par `Defaire` (5 > 3 et 1 > 0), chez la Ligne comme chez la Colonne, tandis que `Defaire` n'est dominée par rien. Le Dilemme du Prisonnier a donc, pour chaque joueur, une stratégie strictement dominée (`Cooperer`) et une stratégie indomptable (`Defaire`) — c'est exactement ce que l'IESDS va énumérer, étape par étape.

## 2. L'IESDS pas-à-pas

La **procédure** IESDS est la suivante : on cherche une stratégie strictement dominée, on la retire du jeu, on recommence sur le jeu réduit, jusqu'à ce qu'aucune stratégie ne soit plus dominée.

Deux points de méthode que la procédure cache :

- l'**ordre** dans lequel on retire les stratégies dominées (section 3) ;
- le fait qu'**éliminer une stratégie peut en créer une nouvelle** : sur le jeu réduit, une stratégie qui n'était pas dominée peut le devenir, parce que la stratégie qui la dominait a disparu.

Le code ci-dessous tient un **journal** des éliminations : chaque étape indique le joueur, la stratégie retirée et la stratégie qui la dominait.

In [3]:
# L'IESDS pas-a-pas, avec journal
def iesds(A, B, labels_l=None, labels_c=None, ordre=None):
    '''Elimination iteree des strategies strictement dominees.

    A, B : matrices de gains (m x n) des deux joueurs.
    labels_l, labels_c : noms des strategies (defaut : R0, R1, ... / C0, C1, ...).
    ordre : priorite d'examen, liste de ("L", index) ou ("C", index). Les index
            sont les index ORIGINAUX des strategies : la priorite reste donc
            stable quand le jeu se reduit.

    Renvoie (survivants_lignes, survivants_colonnes, journal, A_reduit, B_reduit).
    '''
    tous_l = list(labels_l) if labels_l is not None else ["R%d" % i for i in range(A.shape[0])]
    tous_c = list(labels_c) if labels_c is not None else ["C%d" % j for j in range(B.shape[1])]
    Ared = np.array(A, dtype=float)
    Bred = np.array(B, dtype=float)
    actifs_l = list(range(len(tous_l)))
    actifs_c = list(range(len(tous_c)))

    if ordre is None:
        ordre = [("L", i) for i in range(len(tous_l))] + [("C", j) for j in range(len(tous_c))]

    journal = []
    etape = 0
    while True:
        elimine = False
        for joueur, index in ordre:
            if joueur == "L":
                if index not in actifs_l:
                    continue
                pos = actifs_l.index(index)
                dominante = None
                for k in actifs_l:
                    if k != index and np.all(Ared[actifs_l.index(k), :] > Ared[pos, :]):
                        dominante = k
                        break
                if dominante is not None:
                    etape += 1
                    journal.append(
                        "Etape %d : Ligne elimine %s, strictement dominee par %s"
                        % (etape, tous_l[index], tous_l[dominante]))
                    actifs_l.remove(index)
                    Ared = np.delete(Ared, pos, axis=0)
                    Bred = np.delete(Bred, pos, axis=0)
                    elimine = True
                    break
            else:
                if index not in actifs_c:
                    continue
                pos = actifs_c.index(index)
                dominante = None
                for l in actifs_c:
                    if l != index and np.all(Bred[:, actifs_c.index(l)] > Bred[:, pos]):
                        dominante = l
                        break
                if dominante is not None:
                    etape += 1
                    journal.append(
                        "Etape %d : Colonne elimine %s, strictement dominee par %s"
                        % (etape, tous_c[index], tous_c[dominante]))
                    actifs_c.remove(index)
                    Ared = np.delete(Ared, pos, axis=1)
                    Bred = np.delete(Bred, pos, axis=1)
                    elimine = True
                    break
        if not elimine:
            break
    return ([tous_l[i] for i in actifs_l], [tous_c[j] for j in actifs_c],
            journal, Ared, Bred)

### Un jeu en cascade

Le jeu ci-dessous est construit pour que l'élimination soit **en cascade** : une stratégie de la Colonne n'est dominée qu'**après** le retrait d'une stratégie de la Ligne. C'est le point pédagogique central — la dominance se calcule sur le jeu **réduit**, pas sur le jeu initial.

In [4]:
# Jeu en cascade : le retrait de R0 rend C2 strictement dominee
A_cascade = np.array([
    [0, 0, 0],   # R0
    [1, 1, 1],   # R1
    [3, 0, 3],   # R2
], dtype=float)

B_cascade = np.array([
    [0, 0, 0],    # R0
    [2, 1, 1],    # R1
    [0, 3, -1],   # R2
], dtype=float)

jeu_cascade = NormalFormGame(A_cascade, B_cascade,
                              ["R0", "R1", "R2"], ["C0", "C1", "C2"],
                              name="Jeu en cascade")
jeu_cascade.display()

print("Strategies strictement dominees AVANT toute elimination :")
print("  Ligne   :", dominance_stricte_lignes(A_cascade))
print("  Colonne :", dominance_stricte_colonnes(B_cascade))

surv_l, surv_c, journal, A_red, B_red = iesds(A_cascade, B_cascade,
                                              ["R0", "R1", "R2"], ["C0", "C1", "C2"])
print()
print("Journal de l'IESDS :")
for ligne in journal:
    print("  " + ligne)
print()
print("Survivants : lignes", surv_l, "| colonnes", surv_c)
print("Jeu reduit final :")
print(A_red)
print(B_red)


Jeu en cascade
                    C0            C1            C2
--------------------------------------------------
      R0  ( 0.0, 0.0)  ( 0.0, 0.0)  ( 0.0, 0.0)  
      R1  ( 1.0, 2.0)  ( 1.0, 1.0)  ( 1.0, 1.0)  
      R2  ( 3.0, 0.0)  ( 0.0, 3.0)  ( 3.0,-1.0)  
Strategies strictement dominees AVANT toute elimination :
  Ligne   : {0: 1, 1: None, 2: None}
  Colonne : {0: None, 1: None, 2: None}

Journal de l'IESDS :
  Etape 1 : Ligne elimine R0, strictement dominee par R1
  Etape 2 : Colonne elimine C2, strictement dominee par C0

Survivants : lignes ['R1', 'R2'] | colonnes ['C0', 'C1']
Jeu reduit final :
[[1. 1.]
 [3. 0.]]
[[2. 1.]
 [0. 3.]]


### Lecture du résultat

Avant toute élimination, **seule** `R0` est strictement dominée (par `R1`, qui vaut `1 > 0` sur les trois colonnes) ; aucune stratégie de la Colonne ne l'est encore — `C2` vaut `[0, 1, -1]` et `C0` vaut `[0, 2, 0]` : sur la ligne `R0`, les deux sont à égalité (`0`), donc `C0` ne domine **pas** strictement `C2`.

Le journal montre la cascade en deux temps :

1. **R0 est éliminée** par `R1` ;
2. sur le jeu 2×3 restant, `C2` devient strictement dominée par `C0` (`2 > 1` sur `R1`, `0 > -1` sur `R2`) : elle est éliminée à son tour.

Le jeu final est un 2×2 (`R1`, `R2` × `C0`, `C1`). Deux remarques :

- l'**ordre** des deux éliminations n'a pas d'importance ici (elles sont de joueurs différents) — mais qu'en est-il en général ? C'est la question de la section suivante ;
- le jeu réduit **ne dit pas** quel sera l'équilibre : élaguer un jeu, ce n'est pas le résoudre.

## 3. L'indépendance à l'ordre

**Énoncé.** Pour la dominance **stricte**, l'IESDS est *indépendante de l'ordre* : quelle que soit la stratégie dominée retirée en premier, l'ensemble final des survivants est le même.

L'idée de la preuve est simple : la dominance stricte est une relation « pour tout » (sur toutes les stratégies de l'adversaire), donc elle survit au rétrécissement du jeu ; et la relation d'ordre sur les dominateurs est **stricte** (un dominateur strict ne peut pas être lui-même strictement dominé par une stratégie dominée, sans cycle de gains). Le résultat est donc un point fixe, indépendant du chemin.

**Faut-il le croire ?** Non : on le mesure. Le code ci-dessous parcourt **tous** les jeux 2×3 dont les gains sont dans `{0, 1}` — soit $2^6 \times 2^6 = 4096$ jeux — et, pour chacun, compare les survivants obtenus sous plusieurs ordres d'élimination tirés au hasard (graine fixée, donc reproductible).

In [5]:
# Mesure exhaustive : l'IESDS stricte depend-elle de l'ordre ?
def tous_les_jeux_2x3():
    '''Tous les jeux 2x3 dont les gains des deux joueurs sont dans {0, 1}.'''
    for a_flat in itertools.product((0, 1), repeat=6):
        A = np.array(a_flat, dtype=float).reshape(2, 3)
        for b_flat in itertools.product((0, 1), repeat=6):
            B = np.array(b_flat, dtype=float).reshape(2, 3)
            yield A, B


def ordres_aleatoires(rng, m, n, nombre):
    '''Ordres d'examen aleatoires des strategies (lignes et colonnes melangees).'''
    elements = [("L", i) for i in range(m)] + [("C", j) for j in range(n)]
    for _ in range(nombre):
        melange = list(elements)
        rng.shuffle(melange)
        yield melange


rng = np.random.default_rng(0)
nombre_jeux = 0
divergences = []
for A, B in tous_les_jeux_2x3():
    nombre_jeux += 1
    survivants = set()
    for ordre in ordres_aleatoires(rng, 2, 3, 10):
        lig, col, _, _, _ = iesds(A, B, ["R0", "R1"], ["C0", "C1", "C2"], ordre=ordre)
        survivants.add((tuple(lig), tuple(col)))
    if len(survivants) > 1:
        divergences.append((A.copy(), B.copy(), sorted(survivants)))

print("Jeux 2x3 a gains dans {0, 1} analyses : %d" % nombre_jeux)
print("Jeux dont les survivants dependent de l'ordre : %d" % len(divergences))
print("Ordres tires par jeu : 10 (graine 0, reproductible)")

Jeux 2x3 a gains dans {0, 1} analyses : 4096
Jeux dont les survivants dependent de l'ordre : 0
Ordres tires par jeu : 10 (graine 0, reproductible)


### Lecture du résultat

Sur les **4096** jeux 2×3 exhaustivement énumérés, avec dix ordres d'élimination tirés au hasard par jeu, **aucun** jeu ne présente de divergence : l'ensemble des survivants est indépendant de l'ordre. C'est la mesure, pas l'intuition, qui l'établit — et elle est reproductible (graine `0`).

Cette propriété est **spécifique à la dominance stricte**. Avec la dominance *faible* (inégalité large partout, stricte quelque part), l'ordre change le résultat : on le cherche ci-dessous.

### La dominance faible : un ordre qui change tout

Une stratégie $s_i$ est **faiblement dominée** par $s_i'$ si $u_i(s_i', s_{-i}) \geq u_i(s_i, s_{-i})$ pour tout $s_{-i}$, avec inégalité stricte quelque part. La différence avec le cas strict est décisive : la stratégie faiblement dominée peut être **elle-même** la dominatrice d'une autre stratégie. Supprimer la dominée peut donc supprimer sa dominatrice, et l'ordre devient significatif.

Plutôt que d'inventer un exemple à la main, on **cherche** un contre-exemple dans la même famille exhaustive — 4096 jeux 2×3 à gains dans `{0, 1}` — en comparant deux ordres opposés.

In [6]:
# Contre-exemple : l'elimination en dominance FAIBLE depend de l'ordre
def iesds_faible(A, B, labels_l=None, labels_c=None, ordre=None):
    '''Elimination iteree des strategies FAIBLEMENT dominees.

    Une strategie est faiblement dominee s'il existe une autre strategie
    qui fait au moins aussi bien partout, et strictement mieux quelque part.
    Les index de `ordre` sont les index ORIGINAUX des strategies.
    '''
    tous_l = list(labels_l) if labels_l is not None else ["R%d" % i for i in range(A.shape[0])]
    tous_c = list(labels_c) if labels_c is not None else ["C%d" % j for j in range(B.shape[1])]
    Ared = np.array(A, dtype=float)
    Bred = np.array(B, dtype=float)
    actifs_l = list(range(len(tous_l)))
    actifs_c = list(range(len(tous_c)))

    if ordre is None:
        ordre = [("L", i) for i in range(len(tous_l))] + [("C", j) for j in range(len(tous_c))]

    journal = []
    etape = 0
    while True:
        elimine = False
        for joueur, index in ordre:
            if joueur == "L":
                if index not in actifs_l:
                    continue
                pos = actifs_l.index(index)
                dominante = None
                for k in actifs_l:
                    if (k != index
                            and np.all(Ared[actifs_l.index(k), :] >= Ared[pos, :])
                            and np.any(Ared[actifs_l.index(k), :] > Ared[pos, :])):
                        dominante = k
                        break
                if dominante is not None:
                    etape += 1
                    journal.append("Etape %d : Ligne elimine %s (faiblement dominee par %s)"
                                   % (etape, tous_l[index], tous_l[dominante]))
                    actifs_l.remove(index)
                    Ared = np.delete(Ared, pos, axis=0)
                    Bred = np.delete(Bred, pos, axis=0)
                    elimine = True
                    break
            else:
                if index not in actifs_c:
                    continue
                pos = actifs_c.index(index)
                dominante = None
                for l in actifs_c:
                    if (l != index
                            and np.all(Bred[:, actifs_c.index(l)] >= Bred[:, pos])
                            and np.any(Bred[:, actifs_c.index(l)] > Bred[:, pos])):
                        dominante = l
                        break
                if dominante is not None:
                    etape += 1
                    journal.append("Etape %d : Colonne elimine %s (faiblement dominee par %s)"
                                   % (etape, tous_c[index], tous_c[dominante]))
                    actifs_c.remove(index)
                    Ared = np.delete(Ared, pos, axis=1)
                    Bred = np.delete(Bred, pos, axis=1)
                    elimine = True
                    break
        if not elimine:
            break
    return ([tous_l[i] for i in actifs_l], [tous_c[j] for j in actifs_c], journal)


ORDRE_A = [("L", 0), ("L", 1), ("C", 0), ("C", 1), ("C", 2)]
ORDRE_B = list(reversed(ORDRE_A))

contre_exemples = []
for A, B in tous_les_jeux_2x3():
    lig_a, col_a, _ = iesds_faible(A, B, ["R0", "R1"], ["C0", "C1", "C2"], ordre=ORDRE_A)
    lig_b, col_b, _ = iesds_faible(A, B, ["R0", "R1"], ["C0", "C1", "C2"], ordre=ORDRE_B)
    if (lig_a, col_a) != (lig_b, col_b):
        contre_exemples.append((A.copy(), B.copy(), lig_a, col_a, lig_b, col_b))

print("Jeux 2x3 ou l'ordre change le resultat (dominance faible) : %d" % len(contre_exemples))

if contre_exemples:
    A_ex, B_ex, lig_a, col_a, lig_b, col_b = contre_exemples[0]
    print()
    print("Contre-exemple (premier trouve) :")
    print("  Ordre A : L0, L1, C0, C1, C2  ->  lignes", lig_a, "| colonnes", col_a)
    print("  Ordre B : l'ordre inverse       ->  lignes", lig_b, "| colonnes", col_b)
    print()
    print("  Gains de la Ligne (lignes R0, R1) :")
    for i, ligne in enumerate(A_ex):
        print("   R%d" % i, ligne)
    print("  Gains de la Colonne (colonnes C0, C1, C2) :")
    for j in range(B_ex.shape[1]):
        print("   C%d" % j, B_ex[:, j])

Jeux 2x3 ou l'ordre change le resultat (dominance faible) : 1536

Contre-exemple (premier trouve) :
  Ordre A : L0, L1, C0, C1, C2  ->  lignes ['R1'] | colonnes ['C1']
  Ordre B : l'ordre inverse       ->  lignes ['R0', 'R1'] | colonnes ['C1']

  Gains de la Ligne (lignes R0, R1) :
   R0 [0. 0. 0.]
   R1 [0. 0. 1.]
  Gains de la Colonne (colonnes C0, C1, C2) :
   C0 [0. 0.]
   C1 [0. 1.]
   C2 [0. 0.]


### Lecture du résultat

Le compteur **non nul** est le résultat : dans la famille exhaustive des 4096 jeux 2×3 à gains dans `{0, 1}`, il existe des jeux dont l'élimination en dominance faible **dépend de l'ordre**. Le jeu affiché le montre concrètement : les deux ordres ne laissent pas les mêmes stratégies.

La conclusion est donc double, et elle est la même des deux côtés :

- **dominance stricte** : l'ordre n'a aucune importance (mesuré sur 4096 jeux, section précédente) ;
- **dominance faible** : l'ordre a de l'importance (contre-exemple mesuré, ci-dessus).

C'est la raison pour laquelle l'IESDS — et non l'élimination en dominance faible — est la procédure canonique associée aux concepts de solution de la série : elle est **confluente**, donc elle définit un objet (le jeu réduit) et non une procédure.

## 4. Admissibilité et rationalisabilité : la croyance pure ne suffit pas

L'IESDS élimine ce qui est *dominé*. Deux autres notions éliminent ce qui n'est **jamais une meilleure réponse** — et il faut les distinguer, car elles ne coïncident pas.

Une stratégie de la Ligne est **rationalisable au sens ponctuel** si elle est la meilleure réponse (à égalité près) à *au moins une* stratégie **pure** de la Colonne qui survit au processus. On parle de croyance **ponctuelle** : la croyance de l'adversaire est un point, pas une distribution.

Une stratégie est **rationalisable** si elle est la meilleure réponse (à égalité près) à *au moins une croyance **mixte*** — c'est-à-dire à une distribution de probabilités sur les stratégies survivantes de l'adversaire. C'est la définition de Bernheim et Pearce.

L'implication ne joue que dans un sens :

$$\text{rationalisable ponctuellement} \;\Longrightarrow\; \text{rationalisable}$$

Une croyance pure est une croyance mixte dégénérée : qui est la meilleure réponse à la stratégie pure $j$ est la meilleure réponse à la croyance qui met toute la probabilité sur $j$. La réciproque est **fausse** — et c'est le cœur de cette section.

> **Avertissement de vocabulaire.** Le mot *admissible* a un sens **standard** en théorie des jeux : une stratégie est *admissible* si elle n'est **pas faiblement dominée**, ce qui désigne un ensemble **plus grand** que les deux notions ci-dessus. L'employer pour le filtre « meilleure réponse à une croyance pure » exposerait à une confusion : c'est pourquoi le carnet parle de **rationalisabilité ponctuelle**. La fonction `strategies_admissibles` (nom historique, antérieur à ce carnet) en reste l'alias.

### Le contre-exemple : trois lignes, deux colonnes

Considérons le jeu suivant (gains de la Ligne, gains de la Colonne) :

| | C0 | C1 |
|---|---|---|
| **R0** | (3, 1) | (0, 1) |
| **R1** | (0, 1) | (3, 1) |
| **R2** | (2, 0) | (2, 0) |

Regardons **R2** :

- contre la stratégie pure `C0`, les gains de la Ligne sont `3` (R0), `0` (R1), `2` (R2) : R2 est battue sur cette colonne ;
- contre `C1`, les gains sont `0` (R0), `3` (R1), `2` (R2) : R2 est battée là aussi.

R2 n'est donc la meilleure réponse à **aucune** stratégie pure. Mais placez la Colonne sur la croyance $p = (1/2,\ 1/2)$ :

$$u(R0, p) = \tfrac{3}{2} + \tfrac{0}{2} = 1{,}5 \qquad u(R1, p) = 1{,}5 \qquad u(R2, p) = 2$$

R2 est la meilleure réponse à cette **croyance mixte**, et à aucune croyance pure. Les deux notions ne coïncident donc pas : « jamais meilleure réponse à une stratégie pure » n'implique pas « jamais meilleure réponse à un mélange ».

### La question algorithmsique

Pour une stratégie $i$ de gains $u(i, j)$, être la meilleure réponse à une croyance mixte $p$ s'écrit :

$$\exists\; p \geq 0,\quad \sum_j p_j = 1,\quad \forall k:\quad u(i, p) \geq u(k, p)$$

avec $u(i, p) = \sum_j p_j\, u(i, j)$. C'est un **programme linéaire** : on maximise $\varepsilon$ sous les contraintes $(u(i) - u(k)) \cdot p \geq \varepsilon$ pour tout $k$, $\sum_j p_j = 1$ et $p \geq 0$. La stratégie $i$ est rationalisable si et seulement si la valeur optimale de $\varepsilon$ est $\geq 0$ — et elle est toujours $\geq 0$, puisque le choix $k = i$ donne une contrainte nulle : c'est donc l'optimalité qui tranche.

La section 4.1 met cette question en pratique : la meilleure réponse à une croyance mixte est testée par **énumération des sommets** du polyèdre de faisabilité (exacte, et rapide), et confrontée à une **référence par programme linéaire** (`scipy`).


In [7]:
# Admissibilite : elimination iteree des strategies jamais meilleure reponse
def strategies_admissibles(A, B, labels_l=None, labels_c=None):
    '''Elimination iteree des strategies jamais meilleure reponse a une
    strategie PURE de l'adversaire.

    A chaque tour, on compare les strategies SURVIVANTES (jeu restreint) :
    une strategie est retiree si elle n'est la meilleure reponse a aucune
    des strategies adverses survivantes.

    Renvoie (lignes_admissibles, colonnes_admissibles).

    NB TERMINOLOGIQUE : en theorie des jeux, "admissible" a un sens standard
    (strategie NON FAIBLEMENT DOMINEE), qui designe un ensemble PLUS GRAND
    que ce filtre. La notion calculee ici est la "rationalisabilite
    ponctuelle" : etre la meilleure reponse a au moins une croyance PURE.
    Le nom historique de la fonction est conserve en alias plus bas.
    '''
    tous_l = list(labels_l) if labels_l is not None else ["R%d" % i for i in range(A.shape[0])]
    tous_c = list(labels_c) if labels_c is not None else ["C%d" % j for j in range(B.shape[1])]
    actifs_l = list(range(len(tous_l)))
    actifs_c = list(range(len(tous_c)))

    while True:
        # Un joueur sans strategie ne laisse plus rien a comparer : on s'arrete
        # plutot que de vider la liste adverse (le max() n'aurait plus d'argument).
        if not actifs_l or not actifs_c:
            break
        Ar = A[np.ix_(actifs_l, actifs_c)]
        Br = B[np.ix_(actifs_l, actifs_c)]

        def ligne_est_meilleure_reponse(i, j):
            '''La ligne i est-elle meilleure reponse a la colonne j ?'''
            meilleur = max(Ar[p, j] for p in range(len(actifs_l)))
            return Ar[i, j] >= meilleur

        def colonne_est_meilleure_reponse(j, i):
            '''La colonne j est-elle meilleure reponse a la ligne i ?'''
            meilleur = max(Br[i, q] for q in range(len(actifs_c)))
            return Br[i, j] >= meilleur

        a_retirer_l = [p for p in range(len(actifs_l))
                       if not any(ligne_est_meilleure_reponse(p, q)
                                  for q in range(len(actifs_c)))]
        a_retirer_c = [q for q in range(len(actifs_c))
                       if not any(colonne_est_meilleure_reponse(q, p)
                                  for p in range(len(actifs_l)))]
        if not a_retirer_l and not a_retirer_c:
            break
        for p in sorted(a_retirer_l, reverse=True):
            actifs_l.pop(p)
        for q in sorted(a_retirer_c, reverse=True):
            actifs_c.pop(q)
    return ([tous_l[i] for i in actifs_l], [tous_c[j] for j in actifs_c])


# Alias : le filtre se lit "rationalisabilite ponctuelle" (croyance PURE).
# L'ancien nom est conserve pour compatibilite, et renvoie explicitement
# vers le nouveau.
strategies_rationalisables_pures = strategies_admissibles


# La Bataille des Sexes : le classique de la divergence
A_BOS = np.array([[2, 0], [0, 1]], dtype=float)   # Op, Ba
B_BOS = np.array([[1, 0], [0, 2]], dtype=float)   # Op, Ba
jeu_bos = NormalFormGame(A_BOS, B_BOS, ["Opera", "Ballet"], ["Opera", "Ballet"],
                         name="Bataille des Sexes")
jeu_bos.display()

nash_purs = find_pure_nash(jeu_bos)
adm_l, adm_c = strategies_admissibles(A_BOS, B_BOS, ["Opera", "Ballet"], ["Opera", "Ballet"])

print("Equilibres de Nash purs            :", [(jeu_bos.row_labels[i], jeu_bos.col_labels[j])
                                                 for i, j in nash_purs])
print("Strategies rationalisables ponctuelles (Ligne) :", adm_l)
print("Strategies rationalisables ponctuelles (Colonne):", adm_c)

profils_admissibles = [(l, c) for l in adm_l for c in adm_c]
profils_nash = [(jeu_bos.row_labels[i], jeu_bos.col_labels[j]) for i, j in nash_purs]
divergents = [p for p in profils_admissibles if p not in profils_nash]

print()
print("Profils rationalisables ponctuels : %d" % len(profils_admissibles))
print("Profils de Nash          : %d" % len(profils_nash))
print("Rationalisables ponctuels mais PAS de Nash :", divergents)



Bataille des Sexes
                 Opera        Ballet
------------------------------------
   Opera  ( 2.0, 1.0)  ( 0.0, 0.0)  
  Ballet  ( 0.0, 0.0)  ( 1.0, 2.0)  
Equilibres de Nash purs            : [('Opera', 'Opera'), ('Ballet', 'Ballet')]
Strategies rationalisables ponctuelles (Ligne) : ['Opera', 'Ballet']
Strategies rationalisables ponctuelles (Colonne): ['Opera', 'Ballet']

Profils rationalisables ponctuels : 4
Profils de Nash          : 2
Rationalisables ponctuels mais PAS de Nash : [('Opera', 'Ballet'), ('Ballet', 'Opera')]


### Lecture du résultat

Dans la Bataille des Sexes, les **deux** stratégies de chaque joueur survivent au filtre d'admissibilité (`Opera` et `Ballet` pour chacun : chacune est meilleure réponse à la stratégie correspondante de l'adversaire), alors que seuls **deux** profils sont des équilibres de Nash : `(Opera, Opera)` et `(Ballet, Ballet)`.

Le profil `(Opera, Ballet)` est donc **admissible mais pas de Nash** — c'est la divergence recherchée. Elle a une lecture immédiate : après l'élimination, chaque joueur sait que l'adversaire peut jouer `Opera` ou `Ballet` ; mais si les deux jouent `Opera`/`Ballet`, l'un d'eux ferait mieux de changer. « Admissible » dit « je n'exclurai pas ce profil », Nash dit « personne n'y dévie ».

Attention au vocabulaire : ce filtre est l'**admissibilité** (meilleure réponse à une stratégie **pure**). La **rationalisabilité** (meilleure réponse à une croyance **mixte**) est plus permissive — sur ce jeu, elle donne les mêmes survivants, mais la section suivante montre qu'il n'en va pas toujours ainsi.


In [8]:
# Rationalisabilite : etre la meilleure reponse a une croyance MIXTE
#
# Une strategie i est rationalisable s'il existe une distribution p sur les
# strategies survivantes de l'adversaire telle que i soit la meilleure reponse
# a p. La condition s'ecrit :
#
#     (u(i) - u(k)) . p >= 0   pour tout k,   p >= 0,   sum(p) = 1
#
# C'est un programme lineaire. Deux implementations sont donnees :
#  - `meilleure_reponse_melange_lp` : solveur LP (scipy, methode highs).
#    Exact, mais lent : c'est la REFERENCE ;
#  - `meilleure_reponse_melange` : enumeration des sommets du polyedre de
#    faisabilite. Exacte elle aussi (un polyedre non vide a toujours un
#    sommet), et beaucoup plus rapide : c'est elle qui sert a la mesure
#    exhaustive de la section 5.
#
# NB : le cas "croyance unique" (n = 1) est traite a part -- c'est celui ou
# la croyance est deterministe, et il ne se teste qu'en comparant les gains.

TOL = 1e-9


def _contraintes_lignes(gains, i):
    """Contraintes (u(i) - u(k)) . p >= 0, exprimees en coordonnees x.

    On elimine la derniere composante de p par la relation de normalisation :
    p = (x_0, ..., x_{d-1}, 1 - sum(x)) avec d = n - 1. L'equation
    a . p = 0 devient alors (a[:d] - a[d]) . x = -a[d].
    """
    m, n = gains.shape
    d = n - 1
    lignes = []
    for k in range(m):
        if k == i:
            continue
        a = gains[k] - gains[i]
        lignes.append((a[:d] - a[d], -a[d]))
    for j in range(d):
        # x_j >= 0
        e = np.zeros(d)
        e[j] = 1.0
        lignes.append((e, 0.0))
    # sum(x) <= 1, c'est-a-dire p_d = 1 - sum(x) >= 0
    lignes.append((np.ones(d), 1.0))
    return lignes


def _sommets(d, lignes):
    """Sommets candidats du polyedre de faisabilite.

    Un sommet est soit un sommet du simplexe, soit l'intersection de deux
    droites actives (contraintes d'egalite, ou frontieres x_j = 0).
    """
    candidats = [np.eye(d)[j] for j in range(d)] + [np.zeros(d)]
    if d == 1:
        for a, rhs in lignes:
            if abs(a[0]) > 1e-12:
                candidats.append(np.array([rhs / a[0]]))
        return candidats
    for i1 in range(len(lignes)):
        for i2 in range(i1 + 1, len(lignes)):
            M = np.vstack([lignes[i1][0], lignes[i2][0]])
            if M.shape[0] != M.shape[1] or abs(np.linalg.det(M)) < 1e-12:
                continue
            candidats.append(np.linalg.solve(M, np.array([lignes[i1][1], lignes[i2][1]])))
    return candidats


def meilleure_reponse_melange(gains, i):
    """i est-elle la meilleure reponse a UNE croyance mixte ?

    Exact : on enumere les sommets du polyedre de faisabilite et on teste la
    faisabilite de chacun. Le polyedre est non vide si et seulement s'il
    possede un sommet.
    """
    gains = np.asarray(gains, dtype=float)
    m, n = gains.shape
    if n == 1:
        # Croyance unique : i doit battre (ou egaler) toutes les autres.
        return bool(np.all(gains.ravel() <= gains[i, 0] + TOL))
    d = n - 1
    for x in _sommets(d, _contraintes_lignes(gains, i)):
        p = np.concatenate([x, [1.0 - x.sum()]])
        if np.min(p) < -TOL:
            continue
        if all((gains[i] - gains[k]) @ p >= -TOL for k in range(m)):
            return True
    return False


# --- Reference par programme lineaire --------------------------------------
try:
    from scipy.optimize import linprog
    SCIPY_DISPO = True
except ImportError:
    SCIPY_DISPO = False


def meilleure_reponse_melange_lp(gains, i):
    """Meme question, reponse par programme lineaire (reference exacte).

    On maximise eps sous : (u(i) - u(k)) . p >= eps pour tout k,
    sum(p) = 1, p >= 0. La strategie i est rationalisable si la valeur
    optimale d'eps est >= 0.
    """
    if not SCIPY_DISPO:
        raise RuntimeError("scipy absent : la reference LP n'est pas disponible")
    gains = np.asarray(gains, dtype=float)
    m, n = gains.shape
    c = np.zeros(n + 1)
    c[-1] = -1.0
    A_ub = []
    for k in range(m):
        if k == i:
            continue
        ligne = np.zeros(n + 1)
        ligne[:n] = gains[k] - gains[i]
        ligne[-1] = 1.0
        A_ub.append(ligne)
    A_eq = np.concatenate([np.ones(n), [0.0]]).reshape(1, -1)
    res = linprog(c, A_ub=np.array(A_ub), b_ub=np.zeros(len(A_ub)),
                  A_eq=A_eq, b_eq=np.array([1.0]),
                  bounds=[(0, None)] * n + [(None, None)], method="highs")
    return bool(res.status == 0 and res.x[-1] >= -TOL)


def strategies_rationalisables(A, B, labels_l=None, labels_c=None):
    """Elimination iteree des strategies jamais meilleure reponse a une
    croyance MIXTE (Bernheim, Pearce).

    A chaque tour, on compare les strategies SURVIVANTES (jeu restreint) :
    une strategie est retiree s'il n'existe aucune croyance mixte sur les
    strategies adverses survivantes dont elle soit la meilleure reponse.

    Renvoie (lignes_rationalisables, colonnes_rationalisables).
    """
    tous_l = list(labels_l) if labels_l is not None else ["R%d" % i for i in range(A.shape[0])]
    tous_c = list(labels_c) if labels_c is not None else ["C%d" % j for j in range(B.shape[1])]
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    actifs_l = list(range(len(tous_l)))
    actifs_c = list(range(len(tous_c)))

    while True:
        if not actifs_l or not actifs_c:
            break
        Ar = A[np.ix_(actifs_l, actifs_c)]
        Br = B[np.ix_(actifs_l, actifs_c)]

        a_retirer_l = [p for p in range(len(actifs_l))
                       if not meilleure_reponse_melange(Ar, p)]
        a_retirer_c = [q for q in range(len(actifs_c))
                       if not meilleure_reponse_melange(Br.T, q)]
        if not a_retirer_l and not a_retirer_c:
            break
        for p in sorted(a_retirer_l, reverse=True):
            actifs_l.pop(p)
        for q in sorted(a_retirer_c, reverse=True):
            actifs_c.pop(q)
    return ([tous_l[i] for i in actifs_l], [tous_c[j] for j in actifs_c])


# --- Le lemme de Pearce : la meme chose, par la dominance stricte MIXTE ---
# En deux joueurs, l'elimination iteree des strategies "jamais meilleure
# reponse a une croyance mixte" coincide avec l'elimination iteree des
# strategies strictement dominees par un MELANGE de strategies de l'adversaire
# (Pearce 1984). Le test se pose en programme lineaire : on maximise eps sous
#   sum_q q_k u(q, j) >= u(i, j) + eps   pour tout j,   q >= 0,   sum(q) = 1
# et i est strictement dominee si la valeur optimale de eps est strictement
# positive.


def strictement_dominante_mixte(gains, i):
    """i est-elle strictement dominee par un melange des strategies de
    l'adversaire ?

    gains[m, n] : gains de l'adversaire (une colonne = une strategie de
    l'adversaire). On maximise eps tel que le melange q batte i sur TOUTES
    les colonnes, avec une marge eps > 0.
    """
    gains = np.asarray(gains, dtype=float)
    m, n = gains.shape
    c = np.zeros(m + 1)
    c[-1] = -1.0
    A_ub = []
    b_ub = []
    for j in range(n):
        ligne = np.zeros(m + 1)
        ligne[:m] = -(gains[:, j])
        ligne[-1] = 1.0
        A_ub.append(ligne)
        b_ub.append(-gains[i, j])
    A_eq = np.concatenate([np.ones(m), [0.0]]).reshape(1, -1)
    res = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                  A_eq=A_eq, b_eq=np.array([1.0]),
                  bounds=[(0, None)] * m + [(None, None)], method="highs")
    return bool(res.status == 0 and res.x[-1] > 1e-9)


def elimination_dominance_mixte(A, B, labels_l=None, labels_c=None):
    """Elimination iteree des strategies strictement dominees par un melange.

    Renvoie (survivants_lignes, survivants_colonnes).
    """
    tous_l = list(labels_l) if labels_l is not None else ["R%d" % i for i in range(A.shape[0])]
    tous_c = list(labels_c) if labels_c is not None else ["C%d" % j for j in range(B.shape[1])]
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    actifs_l = list(range(len(tous_l)))
    actifs_c = list(range(len(tous_c)))

    while True:
        if not actifs_l or not actifs_c:
            break
        Ar = A[np.ix_(actifs_l, actifs_c)]
        Br = B[np.ix_(actifs_l, actifs_c)]
        a_retirer_l = [p for p in range(len(actifs_l)) if strictement_dominante_mixte(Ar, p)]
        a_retirer_c = [q for q in range(len(actifs_c)) if strictement_dominante_mixte(Br.T, q)]
        if not a_retirer_l and not a_retirer_c:
            break
        for p in sorted(a_retirer_l, reverse=True):
            actifs_l.pop(p)
        for q in sorted(a_retirer_c, reverse=True):
            actifs_c.pop(q)
    return ([tous_l[i] for i in actifs_l], [tous_c[j] for j in actifs_c])


# --- Le contre-exemple de la section 4 -------------------------------------"
A_TMB = np.array([[3, 0], [0, 3], [2, 2]], dtype=float)
B_TMB = np.array([[1, 1], [1, 1], [0, 0]], dtype=float)
jeu_tmb = NormalFormGame(A_TMB, B_TMB, ["R0", "R1", "R2"], ["C0", "C1"],
                         name="Trois lignes, deux colonnes")
jeu_tmb.display()

pure_tmb = [bool(np.any(np.all(A_TMB <= A_TMB[i] + TOL, axis=0))) for i in range(3)]
mixte_tmb = [meilleure_reponse_melange(A_TMB, i) for i in range(3)]
p_milieu = np.array([0.5, 0.5])

print("Meilleure reponse a une croyance PURE  :", pure_tmb)
print("Meilleure reponse a une croyance MIXTE :", mixte_tmb)
print("Gains de la Ligne sur p = (1/2, 1/2)   :", (A_TMB @ p_milieu).tolist())
if SCIPY_DISPO:
    print("Reference LP (scipy)                   :",
          [meilleure_reponse_melange_lp(A_TMB, i) for i in range(3)])
else:
    print("Reference LP (scipy)                   : reference indisponible")

ponc_tmb, poncc_tmb = strategies_rationalisables_pures(A_TMB, B_TMB, ["R0", "R1", "R2"], ["C0", "C1"])
rat_tmb, ratc_tmb = strategies_rationalisables(A_TMB, B_TMB, ["R0", "R1", "R2"], ["C0", "C1"])
print("Rationalisables ponctuels (croyance pure) : lignes %s | colonnes %s" % (ponc_tmb, poncc_tmb))
print("Rationalisables (croyance mixte) : lignes %s | colonnes %s" % (rat_tmb, ratc_tmb))
print("Equilibres de Nash purs         :", [(jeu_tmb.row_labels[i], jeu_tmb.col_labels[j])
                                            for i, j in find_pure_nash(jeu_tmb)])

# --- Confrontation des deux methodes : l'enumeration est-elle exacte ? ------
rng = np.random.default_rng(0)
accords = 0
ecarts = []
if SCIPY_DISPO:
    for _ in range(120):
        A_ech = rng.integers(0, 2, size=(2, 3)).astype(float)
        B_ech = rng.integers(0, 2, size=(2, 3)).astype(float)
        # Les sous-jeux sont confrontes aussi : c'est la que se joue le cas
        # "croyance unique", que le jeu complet ne fait jamais apparaitre.
        for g in (A_ech, B_ech.T, A_ech[:, :1], A_ech[:, :2], B_ech.T[:, :2]):
            for i in range(g.shape[0]):
                r1 = meilleure_reponse_melange(g, i)
                r2 = meilleure_reponse_melange_lp(g, i)
                if r1 == r2:
                    accords += 1
                else:
                    ecarts.append((g.copy(), i, r1, r2))
    print("Confrontation enumeration / LP : %d accords, %d ecarts" % (accords, len(ecarts)))
else:
    print("Confrontation enumeration / LP : reference indisponible (scipy absent)")



Trois lignes, deux colonnes
                    C0            C1
------------------------------------
      R0  ( 3.0, 1.0)  ( 0.0, 1.0)  
      R1  ( 0.0, 1.0)  ( 3.0, 1.0)  
      R2  ( 2.0, 0.0)  ( 2.0, 0.0)  
Meilleure reponse a une croyance PURE  : [True, True, False]
Meilleure reponse a une croyance MIXTE : [True, True, True]
Gains de la Ligne sur p = (1/2, 1/2)   : [1.5, 1.5, 2.0]


Reference LP (scipy)                   : [True, True, True]
Rationalisables ponctuels (croyance pure) : lignes ['R0', 'R1'] | colonnes ['C0', 'C1']
Rationalisables (croyance mixte) : lignes ['R0', 'R1', 'R2'] | colonnes ['C0', 'C1']
Equilibres de Nash purs         : [('R0', 'C0'), ('R1', 'C1')]


Confrontation enumeration / LP : 1440 accords, 0 ecarts


### Lecture du résultat : les deux notions se séparent, et voici pourquoi

Les lignes à lire ensemble disent la même chose que la section précédente, mais **mesurée** :

- **`Meilleure reponse a une croyance PURE`** : `R0` et `R1` y sont (chacune est la meilleure réponse à une colonne où elle vaut `3`), `R2` **non** — battue sur `C0` par `R0` et sur `C1` par `R1` ;
- **`Meilleure reponse a une croyance MIXTE`** : `R2` **y est**, comme `R0` et `R1` ;
- **`Gains de la Ligne sur p = (1/2, 1/2)`** : `1.5`, `1.5`, `2.0` — R2 l'emporte parce que la croyance mixte ne fait pas assez confiance à une colonne pour qu'on la batte ;
- la **référence par programme linéaire** rend le même verdict que l'énumération des sommets : les deux méthodes exactes concordent, y compris sur les sous-jeux qui font apparaître le cas « croyance unique ».

Les deux lignes suivantes comparent les **survivants** des deux procédures d'élimination sur ce même jeu :

- le filtre **ponctuel** (croyance pure) élimine `R2` : elle ne survit qu'à une stratégie **pure** ;
- la **rationalisabilité** (croyance mixte) conserve `R2` : elle survit à une **distribution** de probabilités.

Et les équilibres de Nash du jeu sont `(R0, C0)` et `(R1, C1)` : ils sont rationalisables, comme l'inclusion l'exige.

Le message de la section tient donc en une phrase : **une stratégie peut être irrationnelle au sens de la croyance pure et rationnelle au sens de la croyance mixte**. Éliminer sur le critère « meilleure réponse à une stratégie pure » revient à écarter des stratégies qui ne sont pas irrationnelles — c'est un filtre plus fort, et il ne répond pas à la question que pose Bernheim et Pearce.


## 5. `Nash ⊆ rationalisable` — la mesure exhaustive

L'inclusion se comprend : dans un équilibre de Nash, la stratégie de chaque joueur est la meilleure réponse à celle des autres ; elle est donc la meilleure réponse à une stratégie **pure**, donc à une croyance **mixte** dégénérée, et l'élimination ne la retire jamais.

Mais cette inclusion a un sens **plus fort** qu'elle n'en a l'air : elle doit survivre à toutes les éliminations, y compris celles rendues possibles par les éliminations précédentes. On la vérifie donc sur la même famille exhaustive de 4096 jeux : pour chaque jeu, on compare les équilibres de Nash purs aux profils rationalisables.

On compare au passage les **trois filtres** côte à côte : l'IESDS (éliminer ce qui est *dominé*), l'admissibilité (éliminer ce qui n'est *jamais meilleure réponse à une croyance pure*) et la rationalisabilité (éliminer ce qui n'est *jamais meilleure réponse à une croyance mixte*). L'inclusion attendue est `Rationalisable ⊆ survivants de l'IESDS` — une stratégie strictement dominée n'est jamais une meilleure réponse, elle tombe dès le premier tour. On mesure aussi si les deux filtres de meilleure réponse se distinguent **sur cette famille** : c'est une question de fait, pas de théorème.


In [9]:
# Verification exhaustive : Nash <= rationalisable, et comparaison des filtres
violations_nash = []
violations_inclusion = 0
divergences_iesds = 0
profils_nash_total = 0
profils_rationalisables_total = 0
profils_admissibles_total = 0
jeux_avec_nash = 0
jeux_filtres_identiques = 0

for A, B in tous_les_jeux_2x3():
    jeu = NormalFormGame(A, B, ["R0", "R1"], ["C0", "C1", "C2"])
    nash = find_pure_nash(jeu)
    rat_l, rat_c = strategies_rationalisables(A, B, ["R0", "R1"], ["C0", "C1", "C2"])
    rationalisables = set((l, c) for l in rat_l for c in rat_c)
    adm_l, adm_c = strategies_rationalisables_pures(A, B, ["R0", "R1"], ["C0", "C1", "C2"])
    admissibles = set((l, c) for l in adm_l for c in adm_c)

    profils_nash_total += len(nash)
    profils_rationalisables_total += len(rationalisables)
    profils_admissibles_total += len(admissibles)
    if nash:
        jeux_avec_nash += 1
    for i, j in nash:
        if ("R%d" % i, "C%d" % j) not in rationalisables:
            violations_nash.append((A.copy(), B.copy(), (i, j)))

    # Les deux filtres de meilleure reponse se distinguent-ils sur CETTE famille ?
    if set(rat_l) == set(adm_l) and set(rat_c) == set(adm_c):
        jeux_filtres_identiques += 1

    # Comparaison avec l'IESDS : la rationalisabilite est-elle incluse ?
    iesds_l, iesds_c, _, _, _ = iesds(A, B, ["R0", "R1"], ["C0", "C1", "C2"])
    if not set(rat_l).issubset(set(iesds_l)) or not set(rat_c).issubset(set(iesds_c)):
        violations_inclusion += 1
    # Et l'inverse : l'IESDS garde-t-elle des strategies que la rationalisabilite elimine ?
    if not set(iesds_l).issubset(set(rat_l)) or not set(iesds_c).issubset(set(rat_c)):
        divergences_iesds += 1

print("Jeux analyses                                 : %d" % nombre_jeux)
print("Jeux avec au moins un equilibre de Nash       : %d" % jeux_avec_nash)
print("Profils de Nash rencontres au total          : %d" % profils_nash_total)
print("Profils rationalisables rencontres           : %d" % profils_rationalisables_total)
print("Profils rationalisables ponctuels rencontres : %d" % profils_admissibles_total)
print("Jeux ou les deux filtres coincident          : %d" % jeux_filtres_identiques)
print("Contre-exemples a Nash <= rationalisable     : %d" % len(violations_nash))
print("Contre-exemples a Rationalisable <= IESDS     : %d" % violations_inclusion)
print("Jeux ou l'IESDS garde plus que la rationalisabilite : %d" % divergences_iesds)


Jeux analyses                                 : 4096
Jeux avec au moins un equilibre de Nash       : 4066
Profils de Nash rencontres au total          : 11520
Profils rationalisables rencontres           : 20286
Profils rationalisables ponctuels rencontres : 20286
Jeux ou les deux filtres coincident          : 4096
Contre-exemples a Nash <= rationalisable     : 0
Contre-exemples a Rationalisable <= IESDS     : 0
Jeux ou l'IESDS garde plus que la rationalisabilite : 372


### Lecture du résultat : ce que la famille exhaustive dit des trois filtres

Les compteurs se lisent en quatre temps.

**1. `Nash ⊆ rationalisable` tient.** Le compteur de contre-exemples est **zéro** : sur les 4096 jeux exhaustivement énumérés, chaque profil de Nash est bien un profil rationalisable. Ce n'est donc pas une intuition de manuel — elle tient sur toute la famille, et sa raison est désormais explicite : une stratégie d'un équilibre de Nash est la meilleure réponse à une stratégie **pure** de l'adversaire, donc à une croyance mixte dégénérée.

**2. La divergence est de taille.** 20286 profils rationalisables contre 11520 profils de Nash sur la même famille : la rationalisabilité est strictement plus permissive. Autrement dit, la rationalisabilité est une **approximation** de l'équilibre de Nash — plus facile à vérifier, plus robuste, mais elle ne dit pas quel sera l'équilibre. C'est le sens de l'inclusion :

$$\text{Nash} \subsetneq \text{Rationalisable}$$

**3. Sur cette famille, les deux filtres de meilleure réponse coïncident — et ce n'est pas un théorème.** Le compteur `Profils rationalisables ponctuels` vaut le même total que `Profils rationalisables`, et le compteur `Jeux ou les deux filtres coincident` est de **4096 sur 4096** : sur les jeux 2×3 à gains dans `{0, 1}`, « être la meilleure réponse à une stratégie pure » et « être la meilleure réponse à une croyance mixte » donnent le même ensemble de survivants, jeu après jeu.

C'est un **fait de famille, pas une identité** : la section 4 a mesuré un jeu où les deux notions divergent (`R2` disparaît du filtre ponctuel et survit à la rationalisabilité). La famille 2×3 à gains binaires est trop pauvre pour faire apparaître la différence — il fallait un jeu à trois lignes, dont deux se partagent l'avantage.

**4. Les deux filtres ne sont pas le même filtre — et l'inclusion va dans le sens inverse de ce qu'on attendait.** Le compteur `Rationalisable <= IESDS` est **zéro** : la rationalisabilité ne conserve jamais une stratégie que l'IESDS aurait éliminée (une stratégie strictement dominée n'est jamais une meilleure réponse, elle tombe donc dès le premier tour). Mais le dernier compteur est **non nul** : dans 372 jeux sur 4096, la rationalisabilité élimine une stratégie que la dominance stricte avait conservée.

Autrement dit, **l'IESDS est plus grossière que la rationalisabilité**. Les deux procédures ne sont pas emboîtées l'une dans l'autre : l'IESDS répond à « qui est *dominé* ? » — et il ne teste qu'une stratégie pure comme dominateur —, la rationalisabilité à « qui n'est *jamais optimal* ? », toutes mélangeologies admises. Une stratégie peut être non dominée par une stratégie pure et néanmoins ne jamais être une meilleure réponse : la rationalisabilité la retire, l'IESDS la garde.

C'est la raison mathématique de l'inclusion `Nash ⊆ rationalisable` : seul le filtre par optimalité est **garanti** de conserver tous les équilibres, pas le filtre par dominance.


In [10]:
# Le lemme de Pearce, verifie par echantillon
# En deux joueurs, l'elimination iteree des strategies "jamais meilleure
# reponse a une croyance mixte" doit coincider avec l'elimination iteree des
# strategies strictement dominees par un MELANGE. On le verifie en confrontant
# les deux procedures sur un echantillon de la famille.
#
# ATTENTION : la rationalisabilite (croyance mixte) est calculee ici par
# ENUMERATION DES SOMMETS (rapide) ; la dominance stricte mixte par PROGRAMME
# LINEAIRE (lent). L'echantillon est donc borne -- la verification est un
# echantillon, pas un resultat exhaustif.

rng_lemma = np.random.default_rng(0)
jeux_lemma = 400
ecarts_lemma = []
for _ in range(jeux_lemma):
    A_lem = rng_lemma.integers(0, 2, size=(2, 3)).astype(float)
    B_lem = rng_lemma.integers(0, 2, size=(2, 3)).astype(float)

    rat_l, rat_c = strategies_rationalisables(A_lem, B_lem, ["R0", "R1"], ["C0", "C1", "C2"])
    dom_l, dom_c = elimination_dominance_mixte(A_lem, B_lem, ["R0", "R1"], ["C0", "C1", "C2"])

    if set(rat_l) != set(dom_l) or set(rat_c) != set(dom_c):
        ecarts_lemma.append((A_lem.copy(), B_lem.copy(), rat_l, dom_l, rat_c, dom_c))

print("Echantillon du lemme de Pearce (graine 0) : %d jeux" % jeux_lemma)
print("Jeux ou les deux procedures divergent      : %d" % len(ecarts_lemma))
for A_lem, B_lem, rat_l, dom_l, rat_c, dom_c in ecarts_lemma[:3]:
    print("  ECART : A=%s B=%s | rat %s / %s | dom %s / %s"
          % (A_lem.tolist(), B_lem.tolist(), rat_l, rat_c, dom_l, dom_c))


Echantillon du lemme de Pearce (graine 0) : 400 jeux
Jeux ou les deux procedures divergent      : 0


### Lecture du résultat : le lemme de Pearce, et ce qu'il ne dit pas

Les deux lignes disent la même chose : sur les **400** jeux tirés dans la famille (graine `0`, reproductible), les deux procédures — élimination des stratégies *jamais meilleure réponse à une croyance mixte*, et élimination des stratégies *strictement dominées par un mélange* — donnent **le même ensemble de survivants**, et le compteur de divergences est **zéro**.

C'est le lemme de Pearce : en deux joueurs, les deux filtres sont le même filtre écrit autrement. Il est important de ne pas le confondre avec l'IESDS du carnet, qui teste un dominateur **pur** : c'est précisément pour cela que l'IESDS garde, dans 372 jeux sur 4096, une stratégie que la rationalisabilité retire.

Deux limites à garder en tête :

- la rationalisabilité est calculée par **énumération des sommets** (rapide) et la dominance stricte mixte par **programme linéaire** (lent) : la confrontation est donc bornée à un **échantillon de 400 jeux**, pas aux 4096 jeux de la famille. C'est une vérification par échantillon, pas une preuve ;
- le lemme est un fait de **deux joueurs** : il ne s'étend pas tel quel aux jeux à trois joueurs et plus, où l'argument de la section 4 ne joue plus du tout.


## 6. Exercices

Quatre exercices pour mettre en pratique les outils du carnet. Les solutions ne sont pas fournies : chaque cellule de code est un squelette à compléter (`# TODO`). Toutes les briques nécessaires (`NormalFormGame`, `find_pure_nash`, `dominance_stricte_lignes`, `dominance_stricte_colonnes`, `iesds`, `strategies_rationalisables_pures` — alias historique `strategies_admissibles` —, `strategies_rationalisables`, `meilleure_reponse_melange`, `meilleure_reponse_melange_lp`, `strictement_dominante_mixte`, `elimination_dominance_mixte`, `tous_les_jeux_2x3`) restent disponibles dans l'environnement du carnet.


### Exercice 1 : l'IESDS pas-à-pas sur un jeu 3×3

Reprenez le jeu en cascade du carnet, mais avec des gains **différents** : la dynamique doit être la même — une stratégie de la Colonne ne devient dominée qu'après le retrait d'une stratégie de la Ligne.

| | C0 | C1 | C2 |
|---|---|---|---|
| **R0** | (0, 0) | (0, 0) | (0, 0) |
| **R1** | (2, 3) | (2, 2) | (2, 2) |
| **R2** | (4, 0) | (1, 4) | (4, -2) |

**Objectif** : écrire la fonction `iesds_3x3` qui élimine les stratégies strictement dominées une par une et renvoie le **journal** des éliminations.

- **Indice** : `dominance_stricte_lignes` et `dominance_stricte_colonnes` opèrent sur une sous-matrice — il faut les rejouer sur le jeu réduit, à chaque étape.
- **Etape 1** : trouver la première stratégie strictement dominée (les lignes d'abord).
- **Etape 2** : la retirer, puis recommencer sur le jeu réduit.
- **Etape 3** : renvoyer `(survivants_lignes, survivants_colonnes, journal)` et l'afficher.

In [11]:
# Exercice 1 : l'IESDS pas-a-pas sur un jeu 3x3
# ==================================================================
# TODO : completer `iesds_3x3` (elimination + journal).
# TODO etudiant : renvoyer (survivants_lignes, survivants_colonnes, journal)
resultat_ex1 = None  # TODO etudiant

print("Exercice 1 a completer : la cascade attendue comporte deux eliminations.")

Exercice 1 a completer : la cascade attendue comporte deux eliminations.


### Exercice 2 : l'IESDS dépend-elle de l'ordre ?

Reprenez le jeu en cascade du carnet (`A_cascade`, `B_cascade`) et vérifiez **explicitement** que les survivants ne dépendent pas de l'ordre d'élimination.

**Objectif** : comparer les survivants obtenus sous deux ordres différents, et vérifier qu'ils coïncident.

- **Indice** : la fonction `iesds` accepte un paramètre `ordre`, liste de couples `("L", index)` ou `("C", index)`.
- **Etape 1** : construire un ordre où la Colonne est examinée avant la Ligne, et un autre où la Ligne est examinée avant la Colonne.
- **Etape 2** : comparer les deux résultats et afficher un message explicite si (et seulement si) ils diffèrent.
- **Etape 3** : faire de même avec `iesds_faible` (dominance **faible**) sur le même jeu, et dire si la conclusion change.

In [12]:
# Exercice 2 : l'IESDS depend-elle de l'ordre ?
# ==================================================================
# TODO : comparer les survivants de `iesds` sous deux ordres opposes.
# TODO etudiant : afficher "ordre independant" ou "ordre dependant"
conclusion_ex2 = None  # TODO etudiant

print("Exercice 2 a completer")

Exercice 2 a completer


### Exercice 3 : stratégies rationalisables et croyance mixte

Considérez le jeu suivant (gains de la Ligne, gains de la Colonne) :

| | C0 | C1 |
|---|---|---|
| **R0** | (2, 0) | (0, 1) |
| **R1** | (1, 0) | (1, 0) |

**Objectif** : calculer les stratégies **rationalisables** de ce jeu et les comparer à ses équilibres de Nash.

- **Indice** : `strategies_rationalisables(A, B, labels_l, labels_c)` renvoie les survivants de la rationalisabilité (meilleure réponse à une croyance **mixte**) ; `strategies_rationalisables_pures(A, B, labels_l, labels_c)` renvoie ceux du filtre **ponctuel** (meilleure réponse à une croyance **pure**) ; `find_pure_nash(NormalFormGame(A, B, ...))` renvoie les équilibres purs.
- **Etape 1** : calculer les deux ensembles de survivants et les équilibres de Nash.
- **Etape 2** : vérifier que tout profil de Nash est rationalisable.
- **Etape 3** : afficher les profils rationalisables qui ne sont pas des équilibres de Nash (il y en a au moins un).
- **Etape 4** : sur un jeu **à trois lignes** que vous choisissez, exhibez une stratégie qui survit à la rationalisabilité mais pas au filtre ponctuel — c'est le contre-exemple de la section 4, à refaire par vous-même.


In [13]:
# Exercice 3 : strategies rationalisables
# ==================================================================
A_ex3 = np.array([[2, 0], [1, 1]], dtype=float)
B_ex3 = np.array([[0, 1], [0, 0]], dtype=float)

# TODO : calculer les strategies rationalisables (croyance MIXTE) ET les
# strategies admissibles (croyance PURE), puis les equilibres de Nash purs.
# TODO etudiant : afficher les profils rationalisables qui ne sont PAS de Nash
# TODO etudiant : trouver une strategie (sur un jeu a trois lignes) qui
#                 survit a la rationalisabilite mais pas a l'admissibilite
divergence_ex3 = None  # TODO etudiant

print("Exercice 3 a completer")


Exercice 3 a completer


### Exercice 4 : chercher un contre-exemple en dominance faible

La dominance faible rend l'élimination dépendante de l'ordre (mesuré dans le carnet). Votre travail est de **retrouver** un contre-exemple par vous-même, dans une famille que vous choisissez.

**Objectif** : écrire une fonction `chercher_contre_exemple_faible` qui parcourt une famille de jeux et renvoie le premier jeu dont les survivants diffèrent selon l'ordre.

- **Indice** : `iesds_faible` accepte lui aussi un paramètre `ordre` ; `tous_les_jeux_2x3` fournit une famille exhaustive, mais vous pouvez la réduire (par exemple en fixant `A` et en balayant `B`).
- **Etape 1** : choisir la famille à explorer.
- **Etape 2** : comparer les survivants sous deux ordres opposés.
- **Etape 3** : renvoyer `(A, B, survivants_ordre_1, survivants_ordre_2)` et afficher le jeu trouvé.

In [14]:
# Exercice 4 : chercher un contre-exemple en dominance faible
# ==================================================================
def chercher_contre_exemple_faible(famille=None):
    '''Renvoie (A, B, survivants_ordre_1, survivants_ordre_2) ou None.'''
    # TODO etudiant
    return None


contre_exemple_ex4 = None  # TODO etudiant

print("Exercice 4 a completer")

Exercice 4 a completer


## 7. Résumé

| Concept | Définition | Propriété mesurée |
|---|---|---|
| **Dominance stricte** | $u_i(s_i, s_{-i}) > u_i(s_i', s_{-i})$ pour tout $s_{-i}$ | — |
| **IESDS** | élimination itérée des stratégies strictement dominées (par une stratégie **pure**) | **indépendante de l'ordre** (4096 jeux 2×3, 10 ordres chacun) ; garde **plus** de stratégies que la rationalisabilité (372 jeux sur 4096) |
| **Dominance faible** | $\geq$ partout, $>$ quelque part | **dépendante de l'ordre** (contre-exemple mesuré) |
| **Rationalisabilité ponctuelle** | la stratégie est meilleure réponse à au moins une croyance **pure** survivante | filtre **plus fort** que la rationalisabilité ; n'élimine jamais un équilibre de Nash ; sur la famille 2×3 mesurée, identique à la rationalisabilité |
| **Rationalisabilité** | la stratégie est meilleure réponse à au moins une croyance **mixte** survivante | n'élimine jamais un équilibre de Nash ; coïncide avec l'élimination de la dominance stricte **mixte** (échantillon mesuré) |
| **Profil rationalisable** | produit de stratégies rationalisables | **strictement plus nombreux** que les profils de Nash |

Les deux inclusions à retenir :

$$\text{Nash} \subsetneq \text{Rationalisable} \subseteq \text{IESDS (survivants)}$$

`Nash ⊊ Rationalisable` : 20286 profils rationalisables contre 11520 profils de Nash sur la famille exhaustive, et le profil `(Opera, Ballet)` de la Bataille des Sexes survit au filtre alors qu'il n'est pas un équilibre.

`Rationalisable ⊆ IESDS` : une stratégie strictement dominée n'est jamais une meilleure réponse, elle est donc éliminée par la rationalisabilité dès le premier tour. L'inclusion est **stricte** dans 372 jeux sur 4096 — la rationalisabilité y retire une stratégie que la dominance stricte avait conservée. Les deux filtres ne sont donc pas emboîtés : l'IESDS teste un dominateur **pur**, la rationalisabilité admet un dominateur **mixte**.

Et une relation qu'il ne faut pas confondre avec les deux précédentes :

$$\text{Rationalisable ponctuellement} \;\subseteq\; \text{Rationalisable}$$

Une croyance pure est une croyance mixte dégénérée : l'inclusion est immédiate. Elle est **stricte en général** — le jeu à trois lignes de la section 4 le montre (`R2` survit à la rationalisabilité et pas au filtre ponctuel) — mais **stricte nulle sur la famille mesurée** : sur les 4096 jeux 2×3 à gains dans `{0, 1}`, les deux notions donnent le même ensemble de survivants, jeu après jeu. Une coïncidence de famille, pas une identité.

## Limites

- La mesure exhaustive porte sur les jeux **2×3 à gains dans `{0, 1}`** : c'est une famille complète pour la *structure* des questions posées (ordre, inclusion), mais pas un quelconque jeu 3×3 ou n×m. Elle est de surcroît **trop pauvre** pour séparer la rationalisabilité ponctuelle de la rationalisabilité — il a fallu un jeu à trois lignes, hors famille, pour les distinguer.
- La rationalisabilité est calculée par **énumération des sommets** du polyèdre de faisabilité, confrontée à une **référence par programme linéaire** (`scipy`) : les deux méthodes exactes concordent sur l'échantillon testé, sous-jeux compris. La confrontation est un échantillon, pas une preuve d'égalité sur tout l'espace.
- L'égalité avec l'élimination de la dominance stricte **mixte** (lemme de Pearce) est vérifiée ici sur un **échantillon** de 400 jeux de la famille, et non exhaustivement : le coût du programme linéaire sur les 4096 jeux rendrait l'exécution du carnet trop longue. C'est une **vérification par échantillon**, pas une preuve.
- Le mot *admissible* (sens standard : non faiblement dominée) désigne un ensemble **plus grand** que le filtre ponctuel : le carnet lui préfère donc « rationalisabilité ponctuelle », et `strategies_admissibles` n'est qu'un alias historique.
- Les profils **mixtes** ne sont pas traités ici : l'inclusion `Nash ⊆ rationalisable` est vérifiée sur les équilibres **purs**.

## Pour aller plus loin

- [`GameTheory-02-NormalForm-Python`](GameTheory-02-NormalForm-Python.ipynb) — la forme normale, la dominance, l'IESDS et les équilibres purs
- [`GameTheory-02b-Lean-Definitions-Lean`](GameTheory-02b-Lean-Definitions-Lean.ipynb) — la formalisation Lean des mêmes notions (jeu, stratégie, meilleure réponse, Nash)
- [`GameTheory-04-NashEquilibrium-Python`](GameTheory-04-NashEquilibrium-Python.ipynb) — l'équilibre de Nash en détail, y compris mixte


***

**Notebook précédent** : [GameTheory-02-NormalForm-Python](GameTheory-02-NormalForm-Python.ipynb) · [GameTheory-02b-Lean-Definitions-Lean](GameTheory-02b-Lean-Definitions-Lean.ipynb)
**Notebook suivant** : [GameTheory-04-NashEquilibrium-Python](GameTheory-04-NashEquilibrium-Python.ipynb)